# Quantized SPD-RAG pilot — Kaggle runner

**Research question.** Does hierarchical multi-agent RAG stay robust under aggressive
weight quantization because each document agent solves a small focused problem, or do
small branch-level failures compound as the number of independently required RAG agents
grows?

Three precisions from **one** source checkpoint: `F16`, `Q8_0`, `Q4_K_M`.

**Before you start**
1. Settings → Accelerator → **GPU** (T4 x2 or P100).
2. Settings → Internet → **On** (needed to clone llama.cpp and download the checkpoint;
   if internet is off, attach them as input datasets and use the `--skip-*` flags below).
3. Attach the MultiHop-RAG files (`MultiHopRAG.json`, `corpus.json`) as an input dataset.

**Stage sizes.** Stage A = 18 predictions. Stage B = 36. Stage C = 18. Maximum 72.
The run stops after Stage A unless its gate passes.

**Session budget — a Kaggle GPU notebook is killed at 9 hours of execution**
(the 12 h limit applies to CPU-only sessions). Plan against 9 h:

| Step | Typical | Notes |
| --- | --- | --- |
| deps + preflight | ~5 min | |
| dataset + manifest | ~2 min | must run BEFORE the model build |
| llama.cpp CUDA build | 20–30 min | once per session, not cached across sessions |
| download + F16 conversion | 10–15 min | ~6.2 GB checkpoint |
| importance matrix (Q4_K_M) | 10–25 min | `--no-imatrix` skips it |
| quantize Q8_0 + Q4_K_M | ~5 min | |
| **Stage A** | 30–60 min | 18 predictions |
| **Stage B** | 60–120 min | 36 predictions |
| **Stage C** | 30–60 min | 18 predictions, agent-generated queries |

Everything is resumable: rerun an interrupted cell and it continues from
`results/<stage>/events.jsonl`. If you are close to the wall, stop after Stage B and
run Stage C in a fresh session.

**Disk.** `/kaggle/working` is capped at **20 GB**. The weights alone are ~17.6 GB
before the CUDA build tree, so the model build below passes `--free-hf-weights` to
delete the HF safetensors as soon as the F16 GGUF exists.

## 0. Repo location and CUDA check

In [ ]:
import os, subprocess, sys, json, pathlib

# Point REPO at wherever you put this repository (an input dataset, a git clone,
# or an unzipped upload). It must be writable, so copy it into /kaggle/working.
SRC  = pathlib.Path('/kaggle/input/quantized-spd-rag-pilot')   # <-- edit if needed
REPO = pathlib.Path('/kaggle/working/quantized-spd-rag-pilot')

if not REPO.exists():
    if SRC.exists():
        subprocess.run(['cp','-r',str(SRC),str(REPO)], check=True)
    else:
        raise SystemExit(f'Repository not found. Upload it, then set SRC. Looked in {SRC}')
os.chdir(REPO)
sys.path.insert(0, str(REPO/'src'))

# The results directory is resolved by src/config.py, NOT by this notebook.
# On Kaggle it is /kaggle/working/results, which is NOT `REPO/results` -- every
# relative open('results/...') in this notebook used to raise FileNotFoundError.
from config import results_dir_abs
RESULTS = results_dir_abs()

print('repo   :', REPO)
print('results:', RESULTS)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 1. Dependencies

In [ ]:
# Install the PINNED requirements, not a fresh unbounded resolve. requirements.txt
# carries the bounds this harness is tested against (pydantic<3, sentence-transformers<4).
!pip install -q -r requirements.txt 2>&1 | tail -3
import numpy, pydantic, yaml, sentence_transformers
print('numpy', numpy.__version__, '| pydantic', pydantic.VERSION,
      '| sentence-transformers', sentence_transformers.__version__)

## 2. Preflight — CUDA, input discovery, GPU offload proof

This writes `results/kaggle_preflight.json`. `--skip-preflight` is only for the very
first pass, before any GGUF exists.

In [ ]:
!python scripts/prepare_kaggle.py --skip-preflight

## 3. Dataset → frozen manifest and seed rubric

Deterministic sampling. `data/pilot_manifest.json` is frozen once written, and
`prepare_dataset.py` refuses to overwrite either it or `data/gold_facts.json`
without `--force`.

Questions whose gold answer is literally "yes" or "no" are **excluded** (about 60% of
MultiHop-RAG's answerable questions), because they collapse the answer space at pilot
sample sizes. Pass `--include-yes-no` to keep them. The resulting answer-diversity
statistics are printed below and recorded in the manifest.

**This runs before the model build on purpose**: the Q4_K_M importance matrix is
calibrated on corpus articles *held out* from this manifest, so the manifest has to
exist first.

In [ ]:
!python scripts/prepare_dataset.py --data-dir data --search /kaggle/input

### Review the rubric (recommended before quoting any number)

`data/gold_facts.json` is seeded from the dataset's own evidence facts and every entry
is flagged `needs_manual_review: true`. Reports stay stamped `rubric_reviewed: false`
until you clear them.

In [ ]:
import json
gold = json.load(open('data/gold_facts.json'))
q0 = next(iter(gold['questions'].values()))
print(q0['question'])
print('gold answer:', q0['answer'])
for f in q0['facts']:
    print(' -', f['document_id'], '|', f['text'][:120])

## 4. Build llama.cpp with CUDA, then make the three GGUFs

One conversion path: HF checkpoint → **one** F16 GGUF → `Q8_0` and `Q4_K_M` quantized
from that exact file. Each variant records the sha256 of the F16 it was actually
derived from (`derived_from_f16_sha256`), so `common_source_checkpoint` is checkable
rather than asserted.

`Q4_K_M` is built **with an importance matrix**, calibrated on a deterministic
held-out slice of the corpus — never on the evaluation questions or their supporting
documents. The calibration text, its sha256 and the document ids all land in
`results/provenance.json` alongside `imatrix_used`. `--no-imatrix` reproduces the old
imatrix-less build.

Roughly 45–75 minutes the first time (build + download + convert + imatrix + quantize).
Everything is cached within the session, so re-running the cell is cheap.

*No internet?* Attach the checkpoint as an input dataset and add
`--skip-download --hf-dir /kaggle/input/<your-dataset>`.

In [ ]:
# Free-disk preflight. /kaggle/working is capped at 20 GB and the full build peaks at
# ~21.7 GB unless the HF safetensors are released after conversion, which is why the
# next cell passes --free-hf-weights.
import shutil, pathlib
for target in ('/kaggle/working', str(pathlib.Path.cwd())):
    p = pathlib.Path(target)
    if not p.exists():
        continue
    u = shutil.disk_usage(str(p))
    print(f'{target:24s} free {u.free/2**30:6.2f} GB of {u.total/2**30:7.2f} GB')
print()
print('estimated peak need: ~16.4 GB with --free-hf-weights, ~21.7 GB without')
print('prepare_models.py enforces this with --min-free-gb (default 18);')
print('override deliberately with --allow-low-disk if you know better.')

In [ ]:
!python scripts/prepare_models.py --build-llama-cpp --free-hf-weights --data-dir data --search /kaggle/input

In [ ]:
# Confirm GPU offload with a real load of the smallest model
!python scripts/prepare_kaggle.py --preflight-precision Q4_K_M

## 5. Indexes — one private index per document, isolation proved before any model runs

In [ ]:
!python scripts/build_indexes.py --stage all
# results/ is NOT relative to this repo copy on Kaggle: src/config.py resolves it
# to /kaggle/working/results while cell 2 chdir'd into the repo. Always ask the
# helper, never hardcode a path.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
from config import results_dir_abs
RESULTS = results_dir_abs()

print((RESULTS / 'isolation_report.json').read_text()[:400])

## 6. Stage A — capability calibration (18 predictions)

One precision block at a time: F16, then Q8_0, then Q4_K_M. Question order is
counterbalanced per block and recorded. Resumable — rerun the cell after an
interruption and it picks up where it stopped.

In [ ]:
!python scripts/run_stage.py --stage stage_a --backend llama-server

In [ ]:
!python scripts/score.py --stage stage_a --emit-adjudication
!python scripts/analyze.py --stage stage_a --ni-margin 0.05
!python scripts/package_results.py --stage stage_a

In [ ]:
import json
# results/ is NOT relative to this repo copy on Kaggle: src/config.py resolves it
# to /kaggle/working/results while cell 2 chdir'd into the repo. Always ask the
# helper, never hardcode a path.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
from config import results_dir_abs
RESULTS = results_dir_abs()

gate = json.loads((RESULTS / 'stage_a' / 'gate.json').read_text())
print(json.dumps(gate, indent=2))
print()
print('PROCEED TO STAGE B' if gate['proceed'] else 'STOP — Stage A did not clear its gate')

## 7. Stage B — precision-by-width test (36 predictions)

**Blocked** unless Stage A's gate passed. Add `--force` only as a documented manual
override after reading the Stage A failures.

In [ ]:
!python scripts/run_stage.py --stage stage_b --backend llama-server
!python scripts/score.py --stage stage_b --emit-adjudication
!python scripts/analyze.py --stage stage_b --ni-margin 0.05
!python scripts/package_results.py --stage stage_b

In [ ]:
import json
# results/ is NOT relative to this repo copy on Kaggle: src/config.py resolves it
# to /kaggle/working/results while cell 2 chdir'd into the repo. Always ask the
# helper, never hardcode a path.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
from config import results_dir_abs
RESULTS = results_dir_abs()

a = json.loads((RESULTS / 'stage_b' / 'analysis.json').read_text())
s = a['primary_metric_analysis']
print('scientific outcome :', a['scientific_outcome'])
print('F16-Q4 by width    :', s['f16_q4_gap_by_width'])
print('F16-Q4 overall     :', s['f16_q4_gap_overall'], s['f16_q4_gap_ci'])
print('Q4 verdict         :', s['q4_verdict'])
print('Q8 verdict         :', s['q8_verdict'])

## 8. Stage C — selective end-to-end retrieval (18 predictions)

Six Stage-B questions chosen by a deterministic rule **before** any end-to-end output
exists: 2 low-width, 2 high-width, 2 with the largest fixed-evidence F16/Q4
disagreement. Each document agent now writes its own queries against its private index,
at most two rounds.

If less than ~1 hour of the 9 h session remains, stop here and run Stage C in a fresh
session — the manifest, GGUFs and frozen evidence all reproduce deterministically.

In [ ]:
!python scripts/run_stage.py --stage stage_c --backend llama-server
!python scripts/score.py --stage stage_c
!python scripts/analyze.py --stage stage_c --ni-margin 0.05
!python scripts/package_results.py --stage stage_c

## 9. Final report

`RUN_REPORT.md` says plainly whether real llama.cpp inference completed or only the
harness was exercised. Download the zips from `/kaggle/working/results/`.

The default bundle excludes `adjudication_key.json`. To hand a bundle to a blind human
adjudicator, build it with `package_results.py --stage <stage> --for-adjudication`,
which also drops every precision-labelled artefact.

In [ ]:
from IPython.display import Markdown, display
# results/ is NOT relative to this repo copy on Kaggle: src/config.py resolves it
# to /kaggle/working/results while cell 2 chdir'd into the repo. Always ask the
# helper, never hardcode a path.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
from config import results_dir_abs
RESULTS = results_dir_abs()

for stage in ('stage_a','stage_b','stage_c'):
    p = RESULTS / stage / 'RUN_REPORT.md'
    if p.exists():
        display(Markdown(p.read_text()))
zips = sorted(RESULTS.glob('*.zip'))
print('\n'.join(f'{z}  {z.stat().st_size/2**20:.1f} MiB' for z in zips) or 'no zips yet')

---
### One-command alternative

```bash
python scripts/run_kaggle.py --stage stage_a
python scripts/run_kaggle.py --stage stage_b   # blocked unless Stage A passed
python scripts/run_kaggle.py --stage stage_c
```

### Harness-only check (no GPU, no model, no science)

```bash
python -m pytest tests -q
SPDQ_ALLOW_FIXTURE=1 python scripts/run_stage.py --stage stage_a --backend fixture
```

That path always ends in `SCIENTIFIC_RECOMMENDATION_NOT_AVAILABLE`. By design.